# gridarena LLM: Developer Guide

This notebook is the reference for developers who want to extend or maintain the gridarena LLM system.

Developed and designed by Sebastian Sjøen-Tollaksvik

### What this notebook covers
- How to add a new agent end-to-end: from agent class to dispatcher registration
- How to write and improve RAG documentation so the model actually benefits from it
- How to remove mock data fallbacks when a live database is connected

### Prerequisites
- Familiarity with the gridarena LLM system (see `LLM_tutorial.ipynb`)
- Python 3.10+
- The `lv_grid_llm/` directory checked out and working
- A valid API key for the INESCTEC private server

### Quick orientation — the 6 files every agent touches

| File | Role |
|---|---|
| `agents/your_agent.py` | Agent class: wraps real API calls, falls back to mock |
| `mocks/your_mock.py` | Static fallback data used when DB is not connected |
| `tools/your_tools.py` | Tool schema sent to the LLM: defines function signatures |
| `tools/all_tools.py` | Combines all tool lists into `ALL_TOOLS` |
| `dispatcher.py` | Routes tool calls to agent instances, manages RAG |
| `docs/your_agent.md` | RAG documentation: the model reads this at query time |


---
## 1. Adding a New Agent

This section walks through every step required to add a new agent for a new gridarena API service. The steps must be completed in order — each one depends on the previous.

The example below adds a hypothetical `ForecastAgent` that wraps a `gridarena.routers.forecast` router. Replace `Forecast` / `forecast` with your actual service name throughout.


---
### Step 1: Create the agent class

Create `agents/forecast_agent.py`.

Every agent follows the same pattern without exception:
- One async method per tool
- Every method wraps the real API call in `try/except` and falls back to mock data
- A single `handle()` dispatcher method that routes by `tool_name`
- `rag_context: str = ""` always present on `handle()` even if unused

The `try/except` pattern is not optional. It is what makes the system work without a live database during development and testing.


In [ ]:
# agents/forecast_agent.py
#
# Drop this file into the agents/ directory.
# Replace the imports with your actual gridarena router functions.

import sys, os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))



# Import mock data — the agent must work without a database connection.
from mocks.forecast_mock import FORECAST_MOCK, FORECAST_CONFIG_MOCK

class ForecastAgent:

    def __init__(self):
        self.last_queried_grid = None   # store last grid for conversational context
        self.query_history     = []     # store call history for debugging

    # ── one async method per tool ────────────────────────────────────────────

    async def get_forecast(self, grid_id: str, horizon: str = "24h"):
        self.last_queried_grid = grid_id
        self.query_history.append({"action": "get_forecast", "grid_id": grid_id})
        try:
            return await get_forecast(grid_id=grid_id, horizon=horizon)
        except Exception as e:
            print(f"[Using mock data - reason: {e}]")
            return FORECAST_MOCK

    async def submit_forecast_config(self, grid_id: str, model: str, params: dict):
        self.query_history.append({"action": "submit_forecast_config", "grid_id": grid_id})
        try:
            return await submit_forecast_config(
                grid_id=grid_id,
                model=model,
                params=params
            )
        except Exception as e:
            print(f"[Using mock data - reason: {e}]")
            return FORECAST_CONFIG_MOCK

    # ── handle() is the only entry point the dispatcher calls ───────────────
    # rag_context is always passed by the dispatcher — never remove it.
    # Use tool_args["key"] for required params (KeyError is intentional).
    # Use tool_args.get("key", default) for optional params.

    async def handle(self, tool_name: str, tool_args: dict, rag_context: str = ""):
        if tool_name == "get_forecast":
            return await self.get_forecast(
                grid_id=tool_args["grid_id"],
                horizon=tool_args.get("horizon", "24h")
            )
        elif tool_name == "submit_forecast_config":
            return await self.submit_forecast_config(
                grid_id=tool_args["grid_id"],
                model=tool_args["model"],
                params=tool_args.get("params", {})
            )


**Common mistakes to avoid:**

| Mistake | Consequence |
|---|---|
| `tool_args.get("grid_id")` on a required param | Silently passes `None` to the API, causing a confusing error downstream |
| Missing `rag_context: str = ""` on `handle()` | `TypeError` at runtime: the dispatcher always passes it |
| Raising the exception instead of falling back | System unusable without a live DB during development |



---
### Step 2: Create the mock file

Create `mocks/forecast_mock.py`.

The mock must be a realistic example of the actual API response shape. It is the sole data source when `GRID_DB_DSN` is not set, which covers all development and testing scenarios. If the mock is a placeholder, every response the LLM interprets during development will be wrong.

Look at an existing mock such as `mocks/powerflow_mock.py` or `mocks/phase_mock.py` to understand the expected structure for your router.


In [ ]:
# mocks/forecast_mock.py
#
# These values are returned whenever the real API call fails.
# Make them realistic — the LLM will interpret this data during development.

FORECAST_MOCK = {
    "grid_id": "grid001",
    "horizon": "24h",
    "generated_at": "2025-07-01T00:00:00",
    "forecasts": [
        {
            "timestamp": "2025-07-01T01:00:00",
            "node_id": "PT",
            "predicted_power_active": 125.3,
            "predicted_power_reactive": 12.1,
            "confidence_interval": [118.0, 132.0]
        },
        {
            "timestamp": "2025-07-01T02:00:00",
            "node_id": "PT",
            "predicted_power_active": 119.8,
            "predicted_power_reactive": 11.7,
            "confidence_interval": [112.0, 127.0]
        }
    ]
}

FORECAST_CONFIG_MOCK = {
    "status": "configured",
    "grid_id": "grid001",
    "model": "linear_regression",
    "message": "Forecast model configured successfully."
}


---
### Step 3: Create the tool schema

Create `tools/forecast_tools.py`.

The tool schema is the primary interface between the LLM and your agent. The LLM reads the `description` fields to decide which function to call and what arguments to construct. This means:

- `description` on the function should be an **action phrase**: "Use when the user wants to..." not a noun phrase like "Retrieves forecast data"
- `description` on each parameter should explain what it means in the context of this domain, not just restate its name
- Enum values must exactly match what the API accepts  the LLM will hallucinate values if the enum is wrong or missing
- Put all optional parameters with defaults in the schema  the LLM needs to know they exist


In [ ]:
# tools/forecast_tools.py

FORECAST_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_forecast",
            "description": (
                "Use when the user wants to get a power forecast for a grid, "
                "predict future load, or retrieve upcoming power consumption estimates. "
                "Returns predicted active and reactive power per node per timestamp."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "grid_id": {
                        "type": "string",
                        "description": "The grid ID to retrieve the forecast for"
                    },
                    "horizon": {
                        "type": "string",
                        "enum": ["1h", "6h", "24h", "48h", "7d"],
                        "description": (
                            "Forecast horizon. How far into the future to predict. "
                            "Default is 24h."
                        )
                    }
                },
                "required": ["grid_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_forecast_config",
            "description": (
                "Use when the user wants to configure the forecasting model for a grid, "
                "set a prediction algorithm, or update model parameters."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "grid_id": {
                        "type": "string",
                        "description": "The grid ID to configure the forecast model for"
                    },
                    "model": {
                        "type": "string",
                        "enum": ["linear_regression", "random_forest", "lstm"],
                        "description": "The forecasting model to use"
                    },
                    "params": {
                        "type": "object",
                        "description": "Optional model-specific hyperparameters as a key-value dict"
                    }
                },
                "required": ["grid_id", "model"]
            }
        }
    }
]


---
### Step 4: Register tools in `tools/all_tools.py`

Add your import and include your tool list in `ALL_TOOLS`. The order in `ALL_TOOLS` does not matter — the LLM sees all tools simultaneously.


In [ ]:
# tools/all_tools.py: show the full file with your addition

from tools.grid_tools       import GRID_TOOLS
from tools.historical_tools import HISTORICAL_TOOLS
from tools.phase_tools      import PHASE_TOOLS
from tools.topology_tools   import TOPOLOGY_TOOLS
from tools.state_tools      import STATE_TOOLS
from tools.voltage_tools    import VOLTAGE_TOOLS
from tools.powerflow_tools  import POWERFLOW_TOOLS
from tools.forecast_tools   import FORECAST_TOOLS          # ← add this import

ALL_TOOLS = [
    *GRID_TOOLS,
    *HISTORICAL_TOOLS,
    *PHASE_TOOLS,
    *TOPOLOGY_TOOLS,
    *STATE_TOOLS,
    *VOLTAGE_TOOLS,
    *POWERFLOW_TOOLS,
    *FORECAST_TOOLS,                                        # ← add this line
]


---
### Step 5: Register the agent in `dispatcher.py`

`dispatcher.py` is the central routing file. You need to make **five additions** to it:

1. Import the agent class
2. Add the doc path to `AGENT_DOCS` (used by the embedder)
3. Add keywords to `KEYWORD_TO_AGENT` (used to decide which RAG collections to query)
4. Instantiate the agent
5. Add all tool names to `AGENT_MAP`

Every one of these must be present or the agent will either never be called, never have RAG context, or both.


In [ ]:
# dispatcher.py: show each of the five additions in context

# ── 1. Import ─────────────────────────────────────────────────────────────────
from agents.forecast_agent import ForecastAgent           # ← add this

# ── 2. Add to AGENT_DOCS (for embedder.py) ───────────────────────────────────
AGENT_DOCS = {
    "GridAgent":       SCRIPT_DIR.parent / "docs" / "grid.md",
    "HistoricalAgent": SCRIPT_DIR.parent / "docs" / "historical.md",
    "PhaseAgent":      SCRIPT_DIR.parent / "docs" / "phase.md",
    "TopologyAgent":   SCRIPT_DIR.parent / "docs" / "topology.md",
    "StateAgent":      SCRIPT_DIR.parent / "docs" / "state_estimation.md",
    "VoltageAgent":    SCRIPT_DIR.parent / "docs" / "voltage_control.md",
    "PowerflowAgent":  SCRIPT_DIR.parent / "docs" / "powerflow.md",
    "ForecastAgent":   SCRIPT_DIR.parent / "docs" / "forecast.md",    # ← add this
}

# ── 3. Add keywords for RAG routing ──────────────────────────────────────────
# The dispatcher uses these to decide which ChromaDB collections to query.
# Add all natural language terms an engineer might use when asking about
# this agent's domain. Err on the side of more keywords, not fewer.
KEYWORD_TO_AGENT = {
    # ... existing entries ...
    "forecast":   ["ForecastAgent"],                                   # ← add these
    "predict":    ["ForecastAgent"],
    "prediction": ["ForecastAgent"],
    "future load":["ForecastAgent"],
}

# ── 4. Instantiate the agent ──────────────────────────────────────────────────
forecast_agent = ForecastAgent()                                       # ← add this

# ── 5. Register all tools in AGENT_MAP ───────────────────────────────────────
AGENT_MAP = {
    # ... existing entries ...
    "get_forecast":           forecast_agent,                          # ← add these
    "submit_forecast_config": forecast_agent,
}


**How RAG routing works:** When a query comes in, the dispatcher checks every keyword in `KEYWORD_TO_AGENT` against the query string. Matching agent names are collected and their ChromaDB collections are queried. The 4 most relevant chunks across all matched collections are injected into the system prompt before the LLM sees the query.

If no keywords match, it falls back to `["GridAgent", "VoltageAgent", "PowerflowAgent"]` — so make sure your keywords cover how engineers will phrase questions about your domain.


---
### Step 6: Create the documentation and embed

Create `docs/forecast.md` see **Section 2** for exactly how to write it.

Then re-embed all documentation so the new agent's collection is created in ChromaDB:


Alternatively, INESCTEC.py calls `embed_all_docs()` automatically on startup, so restarting the server achieves the same thing.

The embedder uses 600-word chunks with 100-word overlap. This is by no means a fine tuned number as its just an arbitrary set number during implementation and this is not tested. further development should test and benchmark the parameter tuning for better performance and more effient RAG retrival

 Each agent gets its own ChromaDB collection named after the agent class (e.g. `ForecastAgent`). If the collection already exists it is deleted and recreated there is no manual cleanup needed. 


---
### Step 7: Verify everything works

Test the new agent by asking direct questions about the new agent and see if it calls the correct functions and retrives the right information.


---
### Complete checklist

| Step | File | What to add |
|---|---|---|
| 1 | `agents/forecast_agent.py` | Agent class — one method per tool, `try/except` on every method, `handle()` dispatcher |
| 2 | `mocks/forecast_mock.py` | Realistic static response for each tool |
| 3 | `tools/forecast_tools.py` | Tool schema with action-phrase descriptions and correct enum values |
| 4 | `tools/all_tools.py` | Import + include in `ALL_TOOLS` |
| 5 | `dispatcher.py` | Import, `AGENT_DOCS` entry, keywords in `KEYWORD_TO_AGENT`, instantiation, `AGENT_MAP` entries |
| 6 | `docs/forecast.md` | RAG documentation (see Section 2) |
| 7 | `rag/embedder.py` | just run INESCTEC.py |
| 7 | Direct test | Call `INESCTEC.py` to confirm mock fallback and routing work |


---
## 2. Writing and Improving RAG Documentation

The current markdown files are AI generated and not tested or verified by any experts. All of them should be rewritten in the future following the recipy below:

The markdown files in `docs/` are the only domain knowledge the LLM has beyond the tool schemas. They are embedded into ChromaDB by `rag/embedder.py` and retrieved at query time: the most relevant chunks are injected into the system prompt before every tool call and every interpretation.

**The docs directly determine answer quality.** A well-written doc means the LLM knows voltage thresholds, units, what a score means, and how to interpret output fields. A poorly written doc means the LLM guesses.

There is no upper limit on how detailed these files should be. The only constraint is that detail should be *useful* : it should help the model answer a question better. Exhaustive parameter lists the model can already read from the tool schema are not useful. Concrete thresholds, units, domain context, interpretation guidance, and worked examples are.




---
### 2.1 The core principle: write to match queries, not to describe code

The retriever compares your doc content against the engineer's query using semantic similarity. The closer the phrasing in your doc is to how an engineer would ask a question, the better the retrieval.

**Write prose that sounds like what an engineer would type**, not like API reference documentation.

```
❌  "The get_forecast function accepts a grid_id string and an optional horizon string."

✓   "Use this when you want to get predicted power consumption for a grid over the next
     24 hours. The horizon controls how far ahead you want to look: 24h is the default
     and covers most operational planning use cases."
```

Both convey the same information. The second will retrieve on queries like "how do I predict load for tomorrow" or "what does the forecast function do" because the language matches. This makes the vectordatabase match the query. Which is the parameter used to retrive context


---
### 2.2 Standard structure for every function

Every function section must follow this structure. Do not skip sections: the consistency helps the embedder produce clean chunks.

```markdown
### function_name

Use this when you want to [action the engineer wants to accomplish].
[One sentence on when NOT to use it, if that's non-obvious.]
[One sentence on prerequisites — what must exist before calling this.]

**Parameters:**
- `param_name` (type, required): What it means. Valid values: X, Y, Z. Default: X.
- `optional_param` (type, optional, default=X): What it controls and when you'd change it.

**Returns:** [One line on what the response contains and its structure.]

**How to interpret:**
- `field_name`: [Units, typical range, what a high/low value means.]
- `other_field`: [Thresholds, failure modes, what to look for.]

**Example:**
\```json
{
  "param_name": "example_value",
  "optional_param": "option_a"
}
\```
```

The "How to interpret" section is the most valuable part. The tool schema already tells the model what parameters exist — it does not tell the model what the values mean.


---
### 2.3 What to include in "How to interpret"

Include interpretation guidance whenever any of the following apply:

| Situation | Example |
|---|---|
| The field has a unit | `voltage_magnitude` is in Volts, not per-unit |
| There is a threshold that defines good/bad | Below 207V = undervoltage violation |
| The field is a ratio or score | Accuracy of 1.0 = perfect, random baseline ≈ 0.33 |
| The value is directional | Positive `power_active` = consumption, negative = generation (solar) |
| The field is an error metric | Lower score = better — 0.0 is a perfect state estimate |
| The response contains anonymised IDs | Explain that keys like `anon_key_a1b2` must be preserved exactly in submissions |

**Do not** write interpretation guidance for fields that are self-explanatory from their name, such as `grid_id`, `timestamp`, `status: "ok"`, or `message: "deleted successfully"`.


---
### 2.4 Example — rewriting a weak function section

The following shows what the original `get_power_flow_results` section looked like versus the improved version.


In [ ]:
# ── BEFORE (weak — mirrors the schema, no domain knowledge) ──────────────────

before = """
### get_power_flow_results
Retrieves computed voltages.

Parameters:
- grid_id (string, required)
- phase (string, required)

Returns: Voltage results

formatting:
{
  "grid_id": "grid001",
  "phase": "R"
}

How to interpret:
- voltage.real / imag → complex voltage
- magnitude ≈ sqrt(real² + imag²)
- Used to detect voltage violations
"""

# ── AFTER (strong — gives the model domain knowledge it cannot infer) ─────────

after = """
### get_power_flow_results

Use this when you want to retrieve computed voltages after running a simulation,
check if any node has an undervoltage or overvoltage violation, or analyse the
voltage profile across the grid. Run `run_power_flow` first — results are not
available before the simulation completes.

**Parameters:**
- `grid_id` (string, required): The grid to retrieve results for.
- `phase` (string, required): The phase to retrieve — "R", "S", or "T".

**Returns:** Per-node, per-timestamp complex voltage values.

**How to interpret:**
- `voltage.real` and `voltage.imag`: components of the complex voltage phasor.
  Voltage magnitude = sqrt(real² + imag²) in Volts.
- Nominal voltage for a low-voltage grid is 230V (1.0 per unit).
- Acceptable range: 0.9–1.1 pu 
- Below 207V → undervoltage violation. Typical cause: heavy load or EV charging.
- Above 253V → overvoltage violation. Typical cause: excess solar PV generation.
- Node "PT" is always the reference node and will always read approximately 230V.
- Voltage deviations increase with electrical distance from PT — nodes far from
  the transformer show the largest drops under load.

**Example:**
{
  "grid_id": "grid001",
  "phase": "R"
}
"""

print("Before word count:", len(before.split()))
print("After word count: ", len(after.split()))
print()
print("The 'after' version is longer, but every sentence adds something the model")
print("could not infer from the tool schema alone.")


---
### 2.5 Cross-agent workflow context

Agents do not operate in isolation. The full workflow is:

```
register_grid (GridAgent)
    ↓
register_historical_data (HistoricalAgent)
    ↓
run_power_flow (PowerflowAgent)   
    ↓
get_power_flow_results (PowerflowAgent)
```

**Include this context in your docs.** If a function requires prior steps, say so explicitly in the function's opening sentence. The model does not infer workflow order from schema definitions.

Example — from `powerflow.md`:
```
Use this when you want to simulate the electrical state of the grid.
Prerequisites: the grid must be registered (GridAgent) and historical
data must be uploaded (HistoricalAgent) before this will return results.
```

This phrasing retrieves well on queries like "what do I need before running power flow?" or "why is my simulation failing?".


---
### 2.6 How to verify that RAG context is being retrieved

The dispatcher prints retrieval info on every query. When running INESCTEC.py, look for lines like:

```
[RAG] Found 4 chunks for ForecastAgent
[RAG] Retrieved 312 words of context for ForecastAgent
```

If you see:
```
[RAG] No context found for ForecastAgent
```

It means either: The query keywords did not match any agent in `KEYWORD_TO_AGENT` or the system fell back to default agents that don't include yours.



---
## 3. Removing Mock Data Fallbacks (Production)

During development, every agent falls back to static mock data when the real API call fails. This is intentional: it lets the full LLM pipeline work without a live database.

Once `GRID_DB_DSN` is set and the database is connected, the mock fallback is no longer needed. In production you want real errors to surface: if a grid does not exist, the user should be told, not handed mock data.


---
### 3.1 What to replace

Every agent method currently looks like this:


In [ ]:
# Current pattern — development mode

async def get_forecast(self, grid_id: str, horizon: str = "24h"):
    self.query_history.append({"action": "get_forecast", "grid_id": grid_id})
    try:
        return await get_forecast(grid_id=grid_id, horizon=horizon)
    except Exception as e:
        print(f"[Using mock data - reason: {e}]")
        return FORECAST_MOCK   # ← this is what needs to change


In production, replace the `except` block with one that returns the error so the LLM can communicate it to the user:


In [ ]:
# Production pattern — real errors surfaced to the LLM

async def get_forecast(self, grid_id: str, horizon: str = "24h"):
    self.query_history.append({"action": "get_forecast", "grid_id": grid_id})
    try:
        return await get_forecast(grid_id=grid_id, horizon=horizon)
    except Exception as e:
        return {"error": str(e), "grid_id": grid_id, "function": "get_forecast"}


**Why return the error as a dict rather than raising it?**

The LLM's second call (interpretation) receives the tool output as a string. If you raise the exception, the dispatcher crashes before the LLM can say anything. If you return the error as a dict, the LLM sees `{"error": "Grid grid001 not found"}` and can tell the user in plain English what went wrong.


---
## Summary

| Task | Key files | Command |
|---|---|---|
| Add new agent | `agents/`, `mocks/`, `tools/`, `dispatcher.py`, `docs/` | Follow steps 1–7 in Section 1 |
| Improve RAG docs | `docs/your_agent.md` | Edit then `python rag/embedder.py` |
| Remove mock fallbacks | `agents/your_agent.py` (all) | Replace `except` blocks, verify DB first |

For questions about how to use the system as an end user, see `LLM_tutorial.ipynb`.
